# Extraction Determinism - R11-H119

**Hypothesis**: re-extracting the same 10-document set 5x, entity-set Jaccard variance drops
>=50% under temperature-0 decoding plus a canonicalization prompt (naming rules: fullest form,
no marketing suffixes, singular) versus the production prompt - determinism is substantially a
prompt property, not an inherent LLM limitation.

**Prediction**: naming variance (surface forms) collapses; genuine content variance (which
entities matter) persists at a lower floor - the residual defines the resolver's irreducible workload.

**Acceptance bar**: >=50% variance reduction at EQUAL extraction recall (no entities lost to
rigidity); REFUTED if variance persists.

**Three arms, 5 runs each, identical 10 documents (extraction only - no graph writes)**:
- **Arm A** (production baseline) - production split entity/relation prompt at production temperature
- **Arm B** (treatment) - temperature 0 + canonicalization addendum on the entity-producing prompts
- **Arm C** (attribution) - production prompt at temperature 0, no addendum

**Design note (deviation, recorded honestly)**: the production config for this campaign
(`config-apnea.yml`, and the `LLMSettings` default) already runs the extractor at
**temperature 0.0**. Arm C's configuration is therefore identical to Arm A. There is no
temperature lever to isolate: the temperature effect is zero by construction, so any A->B
reduction is attributable to the prompt. Arm C is still executed as an independent 5-run batch
to empirically confirm A ~= C (the temperature-0 batched-server noise floor).

## GPU selection

Extraction runs on the already-running local vLLM server (gpt-oss-120b, 96GB card, port 8010). This notebook is a client only - it starts no GPU work of its own and imports no torch/tensorflow, so no `CUDA_VISIBLE_DEVICES` selection is needed.

In [1]:
# Imports - grouped by category
from __future__ import annotations
import os                                    # cwd normalization under nbconvert
import json                                  # report + checkpoint serialization
import pickle                                # chunk cache
import time                                  # per-run timing
import itertools                             # run-pair enumeration
import sys                                   # loguru sink
from pathlib import Path                     # filesystem paths
from datetime import datetime, timezone      # UTC report timestamp
from statistics import mean                  # metric aggregation
from collections import Counter              # stable-reference counting

from loguru import logger                    # engine logging (quieted below)
from rapidfuzz import fuzz                    # token_set_ratio for surface/content split

# Project modules (extraction engine, prompts, chunking, models)
from knowledge_graph_foundry.extraction import prompts as prompts_mod
from knowledge_graph_foundry.extraction.extractor import extract_document
from knowledge_graph_foundry.settings import LLMSettings, ExtractionSettings
from knowledge_graph_foundry.engines.local_gpu import LocalGpuEngine
from knowledge_graph_foundry.ingest import read_document
from knowledge_graph_foundry.ingest.chunking import chunk_document
from knowledge_graph_foundry.models import Chunk, Ontology, normalize_name

logger.remove()                              # silence per-chunk DEBUG spam
logger.add(sys.stderr, level="WARNING")

# nbconvert sets the kernel cwd to the notebook's directory; normalize to repo root
# so data/, results/, logs/ resolve correctly.
if Path.cwd().name == "notebooks":
    os.chdir("..")
print("imports ok; cwd:", Path.cwd())

2026-07-07 22:12:01.817 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


imports ok; cwd: /home/lab/workspace/learning/projects/knowledge-graph-foundry


## Configuration

Endpoint, corpus, arms, extraction settings, and output paths are frozen here before any arm runs. The extraction settings mirror production exactly (`split_entity_relation=True`, `gleaning_rounds=1`, chunk 2000/overlap 200); chunk-level request concurrency 8 with 5 documents in flight per arm - the vLLM server batches concurrent requests, so the shared wave-2 ingest interleaves rather than starves.

In [2]:
# --- Frozen configuration ---
ENDPOINT   = "http://localhost:8010/v1"
MODEL      = "gpt-oss-120b"
PURPOSE    = "compare CPAP machines"          # canonical graph purpose (README)
TEMPERATURE = 0.0                             # production temperature (LLMSettings default)
N_RUNS     = 5
CONCURRENCY = 8                               # chunk-level; vLLM batches, wave-2 interleaves fine
CORPUS_DIR = Path("data/external/cpap-datasheets-and-manuals")
CHUNK_CACHE = Path("data/interim/h119_chunks.pkl")
CKPT_DIR   = Path("results/h119")
LOG_PATH   = Path("logs/h119-extraction-determinism.log")
REPORTS_DIR = Path("reports")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Ontology is held FIXED (empty) across every run and arm - a controlled constant so the only
# variables are the prompt text and (nominally) temperature. Production evolves the ontology
# per-document during ingest; freezing it isolates the extraction-prompt effect.
FIXED_ONTOLOGY = Ontology()

# Production extraction settings, verbatim.
EXTRACTION_CFG = ExtractionSettings()         # chunk_size=2000, overlap=200, gleaning_rounds=1, split=True

# Deterministic document selection: sorted order, first 10.
DOCS = [p.name for p in sorted(CORPUS_DIR.glob("*.pdf"))][:10]

ARMS = {
    "A_production":   {"patched": False, "desc": "production prompt, temp 0 (baseline)"},
    "B_canonical":    {"patched": True,  "desc": "production prompt + canonicalization addendum, temp 0"},
    "C_attribution":  {"patched": False, "desc": "production prompt, temp 0 (== A config; noise-floor cross-check)"},
}

def log_line(msg: str) -> None:
    stamp = datetime.now(timezone.utc).strftime("%H:%M:%S")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a") as fh:
        fh.write(line + "\n")

try:
    from rich.console import Console
    from rich.table import Table
    c = Console()
    t = Table(title="H119 configuration", show_lines=False)
    t.add_column("key"); t.add_column("value")
    for k, v in [("endpoint", ENDPOINT), ("model", MODEL), ("purpose", PURPOSE),
                 ("temperature", TEMPERATURE), ("runs/arm", N_RUNS), ("concurrency", CONCURRENCY),
                 ("documents", len(DOCS)), ("arms", ", ".join(ARMS)),
                 ("extraction", f"split={EXTRACTION_CFG.split_entity_relation}, glean={EXTRACTION_CFG.gleaning_rounds}, "
                                f"chunk={EXTRACTION_CFG.chunk_size}/{EXTRACTION_CFG.chunk_overlap}")]:
        t.add_row(k, str(v))
    c.print(t)
    c.print("[bold]documents (sorted, first 10):[/bold]")
    for d in DOCS: c.print(f"  - {d}")
except Exception:
    print("config:", ENDPOINT, MODEL, PURPOSE, "temp", TEMPERATURE, "runs", N_RUNS)
    for d in DOCS: print("  -", d)

                    H119 configuration                    
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ key         ┃ value                                    ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ endpoint    │ http://localhost:8010/v1                 │
│ model       │ gpt-oss-120b                             │
│ purpose     │ compare CPAP machines                    │
│ temperature │ 0.0                                      │
│ runs/arm    │ 5                                        │
│ concurrency │ 8                                        │
│ documents   │ 10                                       │
│ arms        │ A_production, B_canonical, C_attribution │
│ extraction  │ split=True, glean=1, chunk=2000/200      │
└─────────────┴──────────────────────────────────────────┘

documents (sorted, first 10):

- 0-20190113114505.pdf

- 1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pdf

- 3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf

...

- BC-Dreamstation-Standard-CPAP.pdf

- BMC_RESmart_AutoCPAP_User_Manual.pdf

- Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf

- CPAP-Machines-Brochure.pdf

- CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf

## Frozen metric definitions

Every metric is defined here, before any arm runs, so the analysis cannot be tuned to the data.

- **Name normalization** - `models.normalize_name` (strip + lowercase + whitespace-collapse) extended with hyphen->space folding. No H190 glyph normalizer exists in the codebase; this is the stated case/whitespace/hyphen fold.
- **Entity set** per (doc, arm, run) - the set of normalized names across all chunks of the doc
- **Variance** - mean pairwise Jaccard DISTANCE across the 5 runs (10 unordered pairs), averaged over the 10 documents
- **Bar** - `(distance_A - distance_B) / distance_A >= 0.50`
- **Stable reference** per doc - entities appearing in >=3 of arm A's 5 runs; recall retention = fraction of that reference that arm B recovers (appears in >=1 of B's runs). Bar recall clause: >=95%
- **Surface vs content** - for each disagreeing entity in a run pair, if it fuzzy-matches (token_set_ratio >= 80) an entity in the other run it is surface-form variance, else content variance

In [3]:
# --- Frozen metric functions ---
def norm(name: str) -> str:
    base = normalize_name(name)                      # strip + lower + ws-collapse
    return " ".join(base.replace("-", " ").split())  # + hyphen fold

def jaccard_distance(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return 1.0 - len(a & b) / len(a | b)

def mean_pairwise_distance(sets: list) -> float:
    pairs = list(itertools.combinations(range(len(sets)), 2))
    return mean(jaccard_distance(sets[i], sets[j]) for i, j in pairs)

def stable_ref(sets: list, k: int = 3) -> set:
    counts = Counter()
    for s in sets:
        counts.update(s)
    return {e for e, n in counts.items() if n >= k}

def union_set(sets: list) -> set:
    out = set()
    for s in sets: out |= s
    return out

FUZZ_THRESHOLD = 80  # token_set_ratio >= 80 => surface-form counterpart

def surface_content_split(sets_raw: list):
    '''Over all run pairs, classify each disagreeing raw name as surface (fuzzy counterpart
    exists in the other run) or content (no counterpart). Returns (surface, content) counts.'''
    surface = content = 0
    for i, j in itertools.combinations(range(len(sets_raw)), 2):
        ni = {norm(x): x for x in sets_raw[i]}
        nj = {norm(x): x for x in sets_raw[j]}
        only_i = [ni[k] for k in (set(ni) - set(nj))]
        only_j = [nj[k] for k in (set(nj) - set(ni))]
        for names, other in ((only_i, list(nj.values())), (only_j, list(ni.values()))):
            for name in names:
                best = max((fuzz.token_set_ratio(name, o) for o in other), default=0)
                if best >= FUZZ_THRESHOLD:
                    surface += 1
                else:
                    content += 1
    return surface, content

print("metric functions frozen; fuzz threshold =", FUZZ_THRESHOLD)

metric functions frozen; fuzz threshold = 80


## Data loading

Each PDF is parsed once (pymupdf4llm, as the pipeline does) and chunked once (token chunker, 2000/200); the identical chunk text feeds all runs and arms. This isolates extraction (LLM) variance from parse variance. Chunks are cached to `data/interim/h119_chunks.pkl`.

In [4]:
# Load cached chunks (parse once if cache absent).
if CHUNK_CACHE.exists():
    chunk_cache = pickle.loads(CHUNK_CACHE.read_bytes())
    print("loaded chunk cache")
else:
    chunk_cache = {}
    for name in DOCS:
        doc = read_document(CORPUS_DIR / name)
        chunks = chunk_document(doc, chunk_size=EXTRACTION_CFG.chunk_size,
                                chunk_overlap=EXTRACTION_CFG.chunk_overlap)
        chunk_cache[name] = [c.model_dump() for c in chunks]
    CHUNK_CACHE.write_bytes(pickle.dumps(chunk_cache))
    print("parsed and cached chunks")

assert all(d in chunk_cache for d in DOCS), "cache missing a selected document"
CHUNKS = {name: [Chunk(**cd) for cd in chunk_cache[name]] for name in DOCS}
total_chunks = sum(len(v) for v in CHUNKS.values())
for name in DOCS:
    print(f"{len(CHUNKS[name]):2d} chunks  {name}")
print("total chunks:", total_chunks)

loaded chunk cache
 1 chunks  0-20190113114505.pdf
 2 chunks  1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pdf
 8 chunks  3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf
12 chunks  ARTP_Standards_of_Care_-_CPAP_Devices_(Technical_and_Performance)_Version_5.0_-_05-02-2022.pdf
 1 chunks  Airsense-Brochure.pdf
 1 chunks  BC-Dreamstation-Standard-CPAP.pdf
 7 chunks  BMC_RESmart_AutoCPAP_User_Manual.pdf
 1 chunks  Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf
 3 chunks  CPAP-Machines-Brochure.pdf
 1 chunks  CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf
total chunks: 37


## Arm prompts - the canonicalization addendum

Arm B appends a canonicalization addendum to the entity-producing prompts (`_ENTITY_TEMPLATE` and `_GLEANING_TEMPLATE`; the combined `_SYSTEM_TEMPLATE` is patched too for completeness though split extraction does not use it). The relation-only prompt produces no entities and is left untouched. The addendum encodes the registered naming rules: fullest canonical form, strip marketing suffixes, singular nouns, consistent code formatting, and a stable-string requirement.

In [5]:
# Canonicalization addendum (no literal braces - these templates are .format()-ed).
CANON_ADDENDUM = """

Canonicalization rules for entity NAMES (apply to every entity name you emit):
- Use the fullest canonical name as printed - include the manufacturer and the complete model
  designation; never abbreviate or use a partial form
- Strip marketing suffixes and taglines (words like Series, System, Technology, Solution, and
  trademark decoration); keep only the identifying name
- Use singular nouns, not plurals
- Format catalogue and model codes consistently: uppercase letters, no internal spaces
  (write DSX500T11, not dsx500 t11 or DSX500 T11)
- Given the same referent, always emit the byte-identical string across the document"""

_ORIG_TEMPLATES = {
    "_ENTITY_TEMPLATE": prompts_mod._ENTITY_TEMPLATE,
    "_GLEANING_TEMPLATE": prompts_mod._GLEANING_TEMPLATE,
    "_SYSTEM_TEMPLATE": prompts_mod._SYSTEM_TEMPLATE,
}

def set_arm_prompts(patched: bool) -> None:
    """Patch the module-level entity-producing templates for arm B; restore originals otherwise.
    The message-builder functions read these module globals at call time."""
    for key, original in _ORIG_TEMPLATES.items():
        setattr(prompts_mod, key, original + CANON_ADDENDUM if patched else original)

# sanity: patch, check marker, restore
set_arm_prompts(True)
assert prompts_mod._ENTITY_TEMPLATE.endswith("across the document")
set_arm_prompts(False)
assert prompts_mod._ENTITY_TEMPLATE == _ORIG_TEMPLATES["_ENTITY_TEMPLATE"]
print("addendum wired; entity template restored to production form")
print("addendum length:", len(CANON_ADDENDUM), "chars")

addendum wired; entity template restored to production form
addendum length: 634 chars


## Execution

Runs every arm x run x document, capping in-flight requests at 2. Each (arm, run, doc) result is checkpointed to `results/h119/` so a re-execution resumes rather than repeats LLM work. Per-run progress lines go to `logs/h119-extraction-determinism.log`. This is the long cell - expect several hours against the shared endpoint.

In [6]:
# --- Extraction harness (resumable; parallel docs within an arm, arms sequential) ---
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# instructor's v2 mode registry is populated by import side effects: (OPENAI, Mode.JSON) is
# registered only when instructor.v2.providers.openai.handlers is imported, which is
# import-order dependent in an ipykernel. Import it explicitly, then warm up in the main
# thread and serialize construction (filed as DEF-6; production import path happens to work).
import instructor.v2.providers.openai.handlers  # registers (OPENAI, Mode.JSON) handler
_ = LocalGpuEngine(LLMSettings(engine="local-gpu", model=MODEL, base_url=ENDPOINT,
                               temperature=TEMPERATURE, timeout=180))
_engine_lock = threading.Lock()

DOC_WORKERS = 5   # documents in flight per arm; x CONCURRENCY chunks = bounded in-flight requests

def ckpt_path(arm: str, run: int, docname: str) -> Path:
    safe = docname.replace("/", "_")
    return CKPT_DIR / f"{arm}__run{run}__{safe}.json"

def extract_doc(docname: str) -> list:
    '''Return the raw entity names extracted from one document (all chunks).'''
    cfg = LLMSettings(engine="local-gpu", model=MODEL, base_url=ENDPOINT,
                      temperature=TEMPERATURE, timeout=180)
    with _engine_lock:
        engine = LocalGpuEngine(cfg)
    result = extract_document(CHUNKS[docname], PURPOSE, FIXED_ONTOLOGY, engine,
                              concurrency=CONCURRENCY, extraction_cfg=EXTRACTION_CFG)
    return [e.name for e in result.entities]

def _work_item(arm, run, docname):
    cp = ckpt_path(arm, run, docname)
    if cp.exists():
        return arm, run, docname, json.loads(cp.read_text())["names"], None
    t0 = time.time()
    names = extract_doc(docname)
    dt = time.time() - t0
    cp.write_text(json.dumps({"arm": arm, "run": run, "doc": docname,
                              "names": names, "count": len(names),
                              "seconds": round(dt, 1)}))
    return arm, run, docname, names, dt

def run_all():
    data = {arm: {r: {} for r in range(1, N_RUNS + 1)} for arm in ARMS}
    for arm, spec in ARMS.items():
        set_arm_prompts(spec["patched"])   # global patch => arms must not overlap
        log_line(f"=== ARM {arm} ({spec['desc']}) patched={spec['patched']} ===")
        items = [(arm, r, d) for r in range(1, N_RUNS + 1) for d in DOCS]
        with ThreadPoolExecutor(max_workers=DOC_WORKERS) as pool:
            futs = [pool.submit(_work_item, *it) for it in items]
            for f in as_completed(futs):
                a, run, docname, names, dt = f.result()
                data[a][run][docname] = names
                if dt is not None:
                    log_line(f"{a} run{run} {docname[:42]:42s} n={len(names):3d} {dt:6.1f}s")
        set_arm_prompts(False)
    log_line("=== all arms complete ===")
    return data

RAW = run_all()
print("extraction complete")


[20:12:04] === ARM A_production (production prompt, temp 0 (baseline)) patched=False ===


[20:18:20] A_production run1 CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pd n= 27  376.1s


API call failed on attempt 1: litellm.Timeout: APITimeoutError - Request timed out. Error_str: Request timed out.


Max retries exceeded. Total attempts: 1, Last error: litellm.Timeout: APITimeoutError - Request timed out. Error_str: Request timed out.


API call failed on attempt 1: litellm.Timeout: APITimeoutError - Request timed out. Error_str: Request timed out.


...

[00:59:41] C_attribution run5 Brochure_BMC_GIII_A20_Oxygenium_Medical.pd n= 32  606.3s


[00:59:49] C_attribution run5 ARTP_Standards_of_Care_-_CPAP_Devices_(Tec n=179 1059.6s


[00:59:56] C_attribution run5 CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pd n= 18  250.3s


[01:00:27] C_attribution run5 BMC_RESmart_AutoCPAP_User_Manual.pdf       n=185  787.2s


[01:01:17] C_attribution run5 CPAP-Machines-Brochure.pdf                 n= 55  407.2s
[01:01:17] === all arms complete ===
extraction complete


## Metrics

Computes per-doc and per-arm mean pairwise Jaccard distance, the reduction fraction against the bar, recall retention of arm A's stable entities by arm B, the surface-vs-content decomposition, and the arm-C noise-floor cross-check.

In [7]:
# --- Compute metrics from RAW ---
def sets_for(arm: str, docname: str) -> list:
    return [set(norm(x) for x in RAW[arm][run][docname]) for run in range(1, N_RUNS + 1)]

def raw_for(arm: str, docname: str) -> list:
    return [RAW[arm][run][docname] for run in range(1, N_RUNS + 1)]

per_doc_distance = {arm: {} for arm in ARMS}
arm_distance = {}
for arm in ARMS:
    for d in DOCS:
        per_doc_distance[arm][d] = mean_pairwise_distance(sets_for(arm, d))
    arm_distance[arm] = mean(per_doc_distance[arm].values())

raw_counts = {arm: mean(len(RAW[arm][r][d]) for r in range(1, N_RUNS + 1) for d in DOCS)
              for arm in ARMS}

dA = arm_distance["A_production"]
dB = arm_distance["B_canonical"]
dC = arm_distance["C_attribution"]
reduction_B = (dA - dB) / dA if dA else 0.0
reduction_C = (dA - dC) / dA if dA else 0.0

ref_total = inter_total = 0
per_doc_recall = {}
for d in DOCS:
    ref = stable_ref(sets_for("A_production", d), k=3)
    ub = union_set(sets_for("B_canonical", d))
    inter = len(ref & ub)
    per_doc_recall[d] = (inter / len(ref)) if ref else None
    ref_total += len(ref); inter_total += inter
recall_micro = inter_total / ref_total if ref_total else 1.0
recall_macro = mean([v for v in per_doc_recall.values() if v is not None])

surf_A = cont_A = surf_B = cont_B = 0
for d in DOCS:
    s, c = surface_content_split(raw_for("A_production", d)); surf_A += s; cont_A += c
    s, c = surface_content_split(raw_for("B_canonical", d)); surf_B += s; cont_B += c
surface_frac_A = surf_A / (surf_A + cont_A) if (surf_A + cont_A) else 0.0
surface_frac_B = surf_B / (surf_B + cont_B) if (surf_B + cont_B) else 0.0

print(f"arm distances: A={dA:.4f}  B={dB:.4f}  C={dC:.4f}")
print(f"reduction B vs A = {reduction_B:.3f}   (bar >= 0.50)")
print(f"reduction C vs A = {reduction_C:.3f}   (expect ~0: A==C config)")
print(f"raw mean entity counts: A={raw_counts['A_production']:.1f} "
      f"B={raw_counts['B_canonical']:.1f} C={raw_counts['C_attribution']:.1f}")
print(f"recall retention (A-stable recovered by B): micro={recall_micro:.3f} macro={recall_macro:.3f}  (bar >= 0.95)")
print(f"arm A disagreement: surface={surf_A} content={cont_A} (surface frac {surface_frac_A:.3f})")
print(f"arm B disagreement: surface={surf_B} content={cont_B} (surface frac {surface_frac_B:.3f})")

arm distances: A=0.7495  B=0.8220  C=0.6441
reduction B vs A = -0.097   (bar >= 0.50)
reduction C vs A = 0.141   (expect ~0: A==C config)
raw mean entity counts: A=62.0 B=54.9 C=56.4
recall retention (A-stable recovered by B): micro=0.692 macro=0.588  (bar >= 0.95)
arm A disagreement: surface=1810 content=5814 (surface frac 0.237)
arm B disagreement: surface=2898 content=3696 (surface frac 0.439)


## Verdict and report

Evaluates each bar clause and writes the machine-readable report JSON to `reports/`.

In [8]:
# --- Bar evaluation + verdict + report JSON ---
clause_reduction = reduction_B >= 0.50
clause_recall = recall_micro >= 0.95

if clause_reduction and clause_recall:
    verdict = "CONFIRMED"
elif clause_reduction and not clause_recall:
    verdict = "PARTIAL"   # variance dropped but entities lost to rigidity
elif (not clause_reduction) and reduction_B >= 0.20:
    verdict = "PARTIAL"   # meaningful but sub-bar reduction
else:
    verdict = "REFUTED"

ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = {
    "hypothesis": "R11-H119",
    "title": "Extraction determinism - variance is a prompt property",
    "utc": ts,
    "endpoint": ENDPOINT, "model": MODEL, "purpose": PURPOSE,
    "temperature": TEMPERATURE, "runs_per_arm": N_RUNS, "concurrency": CONCURRENCY,
    "documents": DOCS,
    "chunk_counts": {d: len(CHUNKS[d]) for d in DOCS},
    "frozen_metrics": {
        "normalization": "models.normalize_name (strip+lower+ws-collapse) + hyphen->space fold",
        "variance": "mean pairwise Jaccard distance across 5 runs, averaged over 10 docs",
        "bar": "(distance_A - distance_B)/distance_A >= 0.50",
        "stable_reference": "entities in >=3 of arm A's 5 runs",
        "recall_clause": "arm B recovers >=95% of arm-A stable reference (union of B runs)",
        "surface_vs_content": "token_set_ratio >= 80 => surface-form counterpart",
    },
    "extraction_settings": {
        "split_entity_relation": EXTRACTION_CFG.split_entity_relation,
        "gleaning_rounds": EXTRACTION_CFG.gleaning_rounds,
        "chunk_size": EXTRACTION_CFG.chunk_size,
        "chunk_overlap": EXTRACTION_CFG.chunk_overlap,
        "ontology": "empty, fixed across all runs/arms",
    },
    "deviation": ("Production temperature is 0.0 (config-apnea.yml / LLMSettings default). "
                  "Arm C config is identical to Arm A; the temperature lever is zero by "
                  "construction, so any A->B reduction is attributable to the prompt. Arm C run "
                  "as an independent noise-floor cross-check."),
    "per_doc_distance": {arm: per_doc_distance[arm] for arm in ARMS},
    "arm_distance": arm_distance,
    "raw_mean_entity_counts": raw_counts,
    "reduction_B_vs_A": reduction_B,
    "reduction_C_vs_A": reduction_C,
    "recall_retention": {"micro": recall_micro, "macro": recall_macro, "per_doc": per_doc_recall},
    "surface_content": {
        "arm_A": {"surface": surf_A, "content": cont_A, "surface_fraction": surface_frac_A},
        "arm_B": {"surface": surf_B, "content": cont_B, "surface_fraction": surface_frac_B},
    },
    "bar_clauses": {
        "reduction_ge_0.50": clause_reduction,
        "recall_ge_0.95": clause_recall,
    },
    "verdict_recommendation": verdict,
}
out = REPORTS_DIR / f"extraction-determinism-h119-{ts}.json"
out.write_text(json.dumps(report, indent=2))
log_line(f"VERDICT {verdict}  reduction_B={reduction_B:.3f} recall={recall_micro:.3f} -> {out.name}")
print(json.dumps({k: report[k] for k in ("arm_distance", "reduction_B_vs_A", "reduction_C_vs_A",
      "recall_retention", "surface_content", "bar_clauses", "verdict_recommendation")}, indent=2))
print("report ->", out)

[01:01:18] VERDICT REFUTED  reduction_B=-0.097 recall=0.692 -> extraction-determinism-h119-20260708T010118Z.json
{
  "arm_distance": {
    "A_production": 0.7495022455373501,
    "B_canonical": 0.8220133202963247,
    "C_attribution": 0.6440905566748031
  },
  "reduction_B_vs_A": -0.09674564044433019,
  "reduction_C_vs_A": 0.14064225889940182,
  "recall_retention": {
    "micro": 0.6917808219178082,
    "macro": 0.5880518836322408,
    "per_doc": {
      "0-20190113114505.pdf": null,
      "1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pdf": 0.0625,
      "3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf": 0.6493506493506493,
      "ARTP_Standards_of_Care_-_CPAP_Devices_(Technical_and_Performance)_Version_5.0_-_05-02-2022.pdf": 0.8367346938775511,
      "Airsense-Brochure.pdf": null,
      "BC-Dreamstation-Standard-CPAP.pdf": 0.07142857142857142,
      "BMC_RESmart_AutoCPAP_User_Manual.pdf": 0.8409090909090909,
      "Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf": 0.77777777